# Entender `w` y `b`: cómo cambian una predicción y su costo

Este notebook no presupone experiencia previa en estadística. La meta es responder una pregunta simple: **¿por qué una regresión lineal necesita ajustar `w` y `b`?**

Usaremos un ejemplo de horas de estudio (`x`) y calificación (`y`). Una recta será nuestra regla de predicción:

$$\hat{y}=wx+b$$

## 1. Qué representan los parámetros

Imagina que la recta es una regla apoyada sobre una gráfica:

- **`w` (weight o peso)** controla la inclinación. Si `w` aumenta, la recta sube más rápido al avanzar hacia la derecha. Con una única variable, también se llama **pendiente**.
- **`b` (bias o sesgo)** mueve toda la recta hacia arriba o hacia abajo, sin inclinarla. En la gráfica se llama **intercepto** porque es el punto donde la recta corta el eje vertical, es decir, la predicción cuando `x = 0`.

No son intercambiables: cambiar `w` rota la regla; cambiar `b` la desliza verticalmente.

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

# Datos didácticos: cada hora extra aumenta la calificación observada en 3 puntos.
datos = pl.DataFrame({
    "horas_estudio": [0, 1, 2, 3, 4, 5, 6],
    "calificacion_real": [4, 7, 10, 13, 16, 19, 22],
})

def predecir(x, w, b):
    return w * np.asarray(x) + b

def mse(y_real, y_predicho):
    return np.mean((np.asarray(y_real) - np.asarray(y_predicho)) ** 2)

datos

## 2. Cuatro rectas, cuatro comportamientos

En estos datos, la regla perfecta es $\hat y = 3x + 4$. No hace falta adivinarla: en la vida real el algoritmo busca valores parecidos minimizando el costo. Por ahora la usamos para contrastar:

- **Referencia:** `w = 3`, `b = 4`.
- **Peso demasiado pequeño:** `w = 1.5`, `b = 4`. Parte bien, pero se queda corta cada vez más; la recta es demasiado plana.
- **Bias demasiado alto:** `w = 3`, `b = 12`. Tiene la inclinación correcta, pero toda la recta quedó desplazada 8 puntos hacia arriba.
- **Ambos incorrectos:** `w = 4.5`, `b = -2`. Empieza demasiado abajo y sube demasiado rápido.

In [ ]:
modelos = [
    {"nombre": "referencia: w=3, b=4", "w": 3.0, "b": 4.0, "color": "#2ca02c"},
    {"nombre": "w pequeño: w=1.5, b=4", "w": 1.5, "b": 4.0, "color": "#ff7f0e"},
    {"nombre": "b alto: w=3, b=12", "w": 3.0, "b": 12.0, "color": "#d62728"},
    {"nombre": "ambos: w=4.5, b=-2", "w": 4.5, "b": -2.0, "color": "#9467bd"},
]

x = datos["horas_estudio"].to_numpy()
y = datos["calificacion_real"].to_numpy()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y, mode="markers", name="datos reales",
    marker={"size": 10, "color": "black"},
))
for modelo in modelos:
    fig.add_trace(go.Scatter(
        x=x, y=predecir(x, modelo["w"], modelo["b"]), mode="lines",
        name=modelo["nombre"], line={"color": modelo["color"]},
    ))
fig.update_layout(
    title="Cambiar w inclina la recta; cambiar b la desplaza",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

## 3. El costo pone un número al error

Para decidir cuál recta es mejor, no basta con mirarla. Calculamos el **error cuadrático medio**:

$$MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2$$

El MSE es 0 si todas las predicciones coinciden con los datos. Cuanto mayor sea, peor encaja la recta. El cuadrado hace que un error de 8 puntos pese mucho más que uno de 1 punto.

In [ ]:
comparacion = pl.DataFrame([
    {
        "modelo": modelo["nombre"], "w": modelo["w"], "b": modelo["b"],
        "mse": mse(y, predecir(x, modelo["w"], modelo["b"])),
    }
    for modelo in modelos
]).sort("mse")
comparacion

Observa los dos errores de una forma intuitiva:

- Con `w` incorrecto, el error **depende de x**: cerca de 0 puede ser pequeño, pero crece al alejarse. La inclinación no describe bien la relación.
- Con `b` incorrecto, el error es aproximadamente el **mismo desplazamiento vertical** para todos los puntos. La forma es correcta, pero la recta está mal colocada.

Por eso ajustar solo uno no basta normalmente: una predicción eficaz necesita tanto la inclinación como la posición adecuadas.

In [ ]:
# Mostramos los residuos (real - predicción) de dos errores distintos.
residuos = pl.concat([
    datos.select("horas_estudio").with_columns(
        pl.Series("residuo", y - predecir(x, 1.5, 4.0)),
        pl.lit("w pequeño").alias("caso"),
    ),
    datos.select("horas_estudio").with_columns(
        pl.Series("residuo", y - predecir(x, 3.0, 12.0)),
        pl.lit("b alto").alias("caso"),
    ),
])
fig = px.bar(
    residuos, x="horas_estudio", y="residuo", color="caso", barmode="group",
    title="El error por w cambia con x; el de b desplaza todos los puntos",
)
fig.add_hline(y=0, line_color="black")
fig.show()

## 4. Mover un parámetro a la vez

Primero dejamos `b = 4` fijo y probamos muchos valores de `w`. Después dejamos `w = 3` fijo y probamos muchos valores de `b`. Cada gráfica tiene forma de valle: el punto más bajo es el menor costo posible bajo esa condición.

In [ ]:
valores_w = np.linspace(-1, 7, 161)
valores_b = np.linspace(-6, 14, 161)

costo_por_w = pl.DataFrame({
    "w": valores_w,
    "mse": [mse(y, predecir(x, w_candidato, 4)) for w_candidato in valores_w],
})
costo_por_b = pl.DataFrame({
    "b": valores_b,
    "mse": [mse(y, predecir(x, 3, b_candidato)) for b_candidato in valores_b],
})

fig_w = px.line(costo_por_w, x="w", y="mse", title="Costo al cambiar w (b fijo en 4)")
fig_w.add_vline(x=3, line_dash="dash", annotation_text="mejor w")
fig_w.show()

fig_b = px.line(costo_por_b, x="b", y="mse", title="Costo al cambiar b (w fijo en 3)")
fig_b.add_vline(x=4, line_dash="dash", annotation_text="mejor b")
fig_b.show()

## 5. Ajustar `w` y `b` juntos

En realidad el algoritmo prueba ambos parámetros a la vez. Podemos imaginar todos los pares posibles `(w, b)` como un mapa: cada color representa un MSE. La zona más oscura es el mejor par.

El entrenamiento consiste en moverse por este mapa hacia abajo. El descenso de gradiente calcula una dirección que reduce el costo y la sigue en pequeños pasos.

In [ ]:
rejilla_w = np.linspace(0, 6, 81)
rejilla_b = np.linspace(-4, 12, 81)
superficie = pl.DataFrame([
    {"w": w_candidato, "b": b_candidato, "mse": mse(y, predecir(x, w_candidato, b_candidato))}
    for w_candidato in rejilla_w
    for b_candidato in rejilla_b
])

fig = px.density_heatmap(
    superficie, x="w", y="b", z="mse", histfunc="avg",
    color_continuous_scale="Viridis_r",
    title="Mapa de costo: el mínimo está cerca de w=3 y b=4",
)
fig.add_trace(go.Scatter(x=[3], y=[4], mode="markers", name="mejor combinación", marker={"color": "red", "size": 11}))
fig.show()

## 6. Lo esencial para recordar

1. `w` decide **cuánto cambia la predicción** cuando cambia la entrada; es la inclinación.
2. `b` decide **desde qué altura parte** la predicción; es el desplazamiento vertical.
3. Una combinación equivocada de `w` y `b` produce predicciones alejadas de los datos.
4. El MSE convierte esos errores en un único número que podemos minimizar.
5. Entrenar el modelo es encontrar los valores de `w` y `b` con menor costo en los datos de entrenamiento.

**Prueba tú:** cambia los valores de `modelos` y vuelve a ejecutar las gráficas. Antes de mirar el MSE, intenta predecir qué ocurrirá si duplicas `w` o si sumas 5 a `b`.